In [3]:
from neo4j import GraphDatabase
import pandas as pd
from pathlib import Path

# Parametri per la connessione al DB Neo4j locale
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "12345678")

# Creazione del driver 
driver = GraphDatabase.driver(URI, auth=AUTH)

# Test connessione 
def test():
    with driver.session() as session:
        risultato = session.run("RETURN 'stai forte amico' AS messaggio")
        print(risultato.single()["messaggio"])
test()

# Caricamento dataset puliti in memoria RAM
PULITI_PATH = Path('/Users/marcogreco/ComicGraph/Dataset/Puliti')
df_personaggi = pd.read_csv(PULITI_PATH / 'Personaggi_clean.csv')
df_fumetti = pd.read_csv(PULITI_PATH / 'Fumetti_clean.csv')
print(f"Dataset pronti in memoria: {len(df_personaggi):,} personaggi, {len(df_fumetti):,} fumetti.")


stai forte amico
Dataset pronti in memoria: 23,269 personaggi, 60,919 fumetti.


In [7]:
#Creiamo i vincoli di unicità e gli indici per evitare duplicati


def crea_vincoli():
    queries = [
        #ID univoco per ogni personaggio
        "CREATE CONSTRAINT character_id_unique IF NOT EXISTS FOR (c:Character) REQUIRE c.id IS UNIQUE",
        
        #Ogni Fumetto/Albo deve avere issue_id unico
        "CREATE CONSTRAINT issue_id_unique IF NOT EXISTS FOR (i:Issue) REQUIRE i.id IS UNIQUE",

        #Ogni Serie ha un titolo unico
        "CREATE CONSTRAINT series_name_unique IF NOT EXISTS FOR (s:Series) REQUIRE s.name IS UNIQUE",

        # Ogni Autore ha un nome unico
        "CREATE CONSTRAINT creator_name_unique IF NOT EXISTS FOR (cr:Creator) REQUIRE cr.name IS UNIQUE",
        
        # I due publisher (DC e Marvel) sono unici per codice
        "CREATE CONSTRAINT publisher_code_unique IF NOT EXISTS FOR (p:Publisher) REQUIRE p.code IS UNIQUE",
    ]


    with driver.session() as session: 
        for q in queries:
            session.run(q)
            print(f"eseguito: {q.split()[2]}")
    print("vincoli creati")
crea_vincoli()

eseguito: character_id_unique
eseguito: issue_id_unique
eseguito: series_name_unique
eseguito: creator_name_unique
eseguito: publisher_code_unique
vincoli creati


In [ ]:
#Creazione dei Publishers (editori) cioè i nodi radice

query_publishers = """ 
MERGE (dc:Publisher {code: 'DC'})
SET dc.name = 'DC Comics'

MERGE (marvel:Publisher {code: 'Marvel'})
SET marvel.name = 'Marvel Comics'
"""


with driver.session() as session: 
    session.run(query_publishers)
    print("editori creati")

editori creati


In [ ]:
#Creazione dei personaggi 

#caricamento del dataset personaggi
PULITI_PATH = Path('/Users/marcogreco/ComicGraph/Dataset/Puliti')
df_personaggi = pd.read_csv(PULITI_PATH / 'Personaggi_clean.csv')


#gestione dei valori mancanti
df_personaggi = df_personaggi.astype(object).where(pd.notnull(df_personaggi), None)


#Query Cypher: creiazione del nodo Character e la relazione PUBLISHED_BY con il Publisher
# Query Cypher: creazione del nodo Character e relazione PUBLISHED_BY con il Publisher
query_carica_personaggi = """
UNWIND $batch AS row
MERGE (c:Character {id: row.character_id})
SET c.name = row.name,
    c.original_page_id = toInteger(row.original_page_id),
    c.url_slug = row.url_slug,
    c.identity_status = row.identity_status,
    c.alignment = row.alignment,
    c.eye_color = row.eye_color,
    c.hair_color = row.hair_color,
    c.gender = row.gender,
    c.status = row.status,
    c.appearances_count = toInteger(row.appearances_count),
    c.first_appearance_year = toInteger(row.first_appearance_year),
    c.wiki_url = row.wiki_url
WITH c, row
MATCH (p:Publisher {code: row.publisher})
MERGE (c)-[:PUBLISHED_BY]->(p)
"""

# 3. Inserimento a batch
batch_size = 2000
totale = len(df_personaggi)
with driver.session() as session:
    for i in range(0, totale, batch_size):
        batch = df_personaggi.iloc[i : i + batch_size].to_dict(orient="records")
        session.run(query_carica_personaggi, batch=batch)
        print(f"Caricati {min(i + batch_size, totale)} / {totale} personaggi...")
print("Personaggi collegati ai publisher")



Caricati 2000 / 23269 personaggi...
Caricati 4000 / 23269 personaggi...
Caricati 6000 / 23269 personaggi...
Caricati 8000 / 23269 personaggi...
Caricati 10000 / 23269 personaggi...
Caricati 12000 / 23269 personaggi...
Caricati 14000 / 23269 personaggi...
Caricati 16000 / 23269 personaggi...
Caricati 18000 / 23269 personaggi...
Caricati 20000 / 23269 personaggi...
Caricati 22000 / 23269 personaggi...
Caricati 23269 / 23269 personaggi...
Personaggi collegati ai publisher


In [10]:
#Caricamento fumetti 

#caricamento dataset 
df_fumetti = pd.read_csv(PULITI_PATH / 'Fumetti_clean.csv') 

#gestione valore mancanti
df_fumetti = df_fumetti.astype(object).where(pd.notnull(df_fumetti), None)



#Query completa con nodi Issue Series Creator ( se ci sono più creator vengono separati)
query_carica_fumetti = """
UNWIND $batch AS row
// 1. Crea il nodo Issue
MERGE (i:Issue {id: row.issue_id})
SET i.title = row.issue_title,
    i.release_year = toInteger(row.release_year),
    i.comic_type = row.comic_type
// 2. Crea la Serie e collega l'Issue
MERGE (s:Series {name: row.series_title})
MERGE (i)-[:PART_OF_SERIES]->(s)
// 3. Collega l'Issue al Publisher (DC o Marvel)
WITH i, row
MATCH (p:Publisher {code: row.publisher})
MERGE (i)-[:PUBLISHED_BY]->(p)
// 4. Autori (Writers)
FOREACH (w IN CASE WHEN row.writer IS NOT NULL THEN [x IN split(row.writer, ",") WHERE trim(x) <> ""] ELSE [] END |
    MERGE (cr:Creator {name: trim(w)})
    MERGE (cr)-[:WROTE]->(i)
)
// 5. Disegnatori (Pencilers)
FOREACH (pen IN CASE WHEN row.penciler IS NOT NULL THEN [x IN split(row.penciler, ",") WHERE trim(x) <> ""] ELSE [] END |
    MERGE (cr:Creator {name: trim(pen)})
    MERGE (cr)-[:PENCILED]->(i)
)
// 6. Cover Artists
FOREACH (art IN CASE WHEN row.cover_artist IS NOT NULL THEN [x IN split(row.cover_artist, ",") WHERE trim(x) <> ""] ELSE [] END |
    MERGE (cr:Creator {name: trim(art)})
    MERGE (cr)-[:COVER_ARTIST_FOR]->(i)
)
"""

# 3. Caricamento a batch da 2.000 righe
batch_size = 2000
totale_fumetti = len(df_fumetti)
with driver.session() as session:
    for i in range(0, totale_fumetti, batch_size):
        batch = df_fumetti.iloc[i : i + batch_size].to_dict(orient="records")
        session.run(query_carica_fumetti, batch=batch)
        print(f"Caricati {min(i + batch_size, totale_fumetti)} / {totale_fumetti} albi con autori e serie...")
print("\n tutte le query sono state caricate")


Caricati 2000 / 60919 albi con autori e serie...
Caricati 4000 / 60919 albi con autori e serie...
Caricati 6000 / 60919 albi con autori e serie...
Caricati 8000 / 60919 albi con autori e serie...
Caricati 10000 / 60919 albi con autori e serie...
Caricati 12000 / 60919 albi con autori e serie...
Caricati 14000 / 60919 albi con autori e serie...
Caricati 16000 / 60919 albi con autori e serie...
Caricati 18000 / 60919 albi con autori e serie...
Caricati 20000 / 60919 albi con autori e serie...
Caricati 22000 / 60919 albi con autori e serie...
Caricati 24000 / 60919 albi con autori e serie...
Caricati 26000 / 60919 albi con autori e serie...
Caricati 28000 / 60919 albi con autori e serie...
Caricati 30000 / 60919 albi con autori e serie...
Caricati 32000 / 60919 albi con autori e serie...
Caricati 34000 / 60919 albi con autori e serie...
Caricati 36000 / 60919 albi con autori e serie...
Caricati 38000 / 60919 albi con autori e serie...
Caricati 40000 / 60919 albi con autori e serie...
Cari

In [4]:
#Mette in relazione i personaggi con le serie

import re 
import time


#Estrae Alter ego e nome reale 
def estrai_varianti_nome (nome_completo):
    m = re.match(r"^(.*?)\s*\((.*?)\)$", str(nome_completo).strip())
    if m: 
        alias = m.group(1).strip()
        inside = m.group(2).strip()

        #Evita i tag derivanti dalla dimensione di provenienza 
        if re.search(r"Earth|Universe|Prime|Timeline|Dimension|Era", inside, re.IGNORECASE):
            return [alias]
        #Legge i soprannomi tra virgolette
        nicknames = re.findall(r"\"(.*?)\"", inside)
        nome_civile = re.sub(r"\".*?\"", "", inside).strip()
        nome_civile = re.sub(r"\s+", " ", nome_civile)
        
        varianti = [alias]
        if nome_civile and len(nome_civile) >= 3:
            varianti.append(nome_civile)
        for nick in nicknames:
            if len(nick) >= 3:
                varianti.append(nick)
        return varianti
    return [str(nome_completo).strip()]

#Parole generiche per evitare falsi positivi
STOPWORDS = {"the", "and", "man", "boy", "girl", "one", "two", "red", "all", "new", "war", "she", "mr.", "ms.", "he", "it", "who"}

#Ordinamento per numero di presenze decrescenti
df_personaggi_sorted = df_personaggi.sort_values(by="appearances_count", ascending=False)

#tabella lookup collega id al personaggio 
varianti = []
for _, r in df_personaggi_sorted.iterrows():
    c_id = r["character_id"]
    pub = r["publisher"]
    for n in estrai_varianti_nome(r["name"]):
        if len(n) >= 3 and n.lower() not in STOPWORDS:
            varianti.append({"character_id": c_id, "publisher": pub, "name_variant": n})

df_varianti = pd.DataFrame(varianti).drop_duplicates(subset=["publisher", "name_variant"])

#Estrae le serie uniche per publisher
dc_series = df_fumetti[df_fumetti["publisher"] == "DC"]["series_title"].drop_duplicates().tolist()
marvel_series = df_fumetti[df_fumetti["publisher"] == "Marvel"]["series_title"].drop_duplicates().tolist()

#Matching con regex e confini 
def match_publisher_series(df_var_pub, series_list):
    name_to_id = dict(zip(df_var_pub["name_variant"].str.lower(), df_var_pub["character_id"]))
    sorted_names = sorted(df_var_pub["name_variant"].tolist(), key=len, reverse=True)
    
    matches = []
    chunk_size = 400
    for i in range(0, len(sorted_names), chunk_size):
        chunk = sorted_names[i : i + chunk_size]
        pattern = re.compile(r"\b(" + "|".join(re.escape(x) for x in chunk) + r")\b", re.IGNORECASE)
        for s in series_list:
            found = pattern.findall(s)
            for item in set(found):
                cid = name_to_id.get(item.lower())
                if cid:
                    matches.append((cid, s))
    return matches
t0 = time.time()
m_dc = match_publisher_series(df_varianti[df_varianti["publisher"] == "DC"], dc_series)
m_marvel = match_publisher_series(df_varianti[df_varianti["publisher"] == "Marvel"], marvel_series)

#Collega superman all'antologia di Action Comics e Batman a Detective Comics
superman_id = df_personaggi[df_personaggi["name"].str.startswith("Superman (Clark Kent)")]["character_id"].iloc[0]
batman_id = df_personaggi[df_personaggi["name"].str.startswith("Batman (Bruce Wayne)")]["character_id"].iloc[0]
for s in dc_series:
    if re.search(r"\bAction Comics\b", s, re.IGNORECASE):
        m_dc.append((superman_id, s))
    elif re.search(r"\bDetective Comics\b", s, re.IGNORECASE):
        m_dc.append((batman_id, s))

# Gestione speciale Punisher: rimuove il Cyborg di Galactus e assegna le serie a Frank Castle
frank_castle_id = df_personaggi[df_personaggi["name"].str.startswith("Frank Castle")]["character_id"].iloc[0]
cyborg_id = df_personaggi[df_personaggi["name"].str.startswith("Punisher (Cyborg)")]["character_id"].iloc[0]
m_marvel = [(cid, s) for cid, s in m_marvel if cid != cyborg_id]
for s in marvel_series:
    if re.search(r"\bpunisher\b", s, re.IGNORECASE):
        m_marvel.append((frank_castle_id, s))

tutti_collegamenti = list(set(m_dc + m_marvel))
t1 = time.time()
print(f"Calcolati {len(tutti_collegamenti):,} collegamenti")

#Caricamento in Neo4j
query_collega = """
UNWIND $batch AS row
MATCH (c:Character {id: row.character_id})
MATCH (s:Series {name: row.series_name})
MERGE (c)-[:STARS_IN]->(s)
"""
df_collegamenti = pd.DataFrame(tutti_collegamenti, columns=["character_id", "series_name"])
batch_size = 2000
totale_link = len(df_collegamenti)
with driver.session() as session:
    for i in range(0, totale_link, batch_size):
        batch = df_collegamenti.iloc[i : i + batch_size].to_dict(orient="records")
        session.run(query_collega, batch=batch)
        print(f"Salvati {min(i + batch_size, totale_link)} / {totale_link} ")
print("\nRelazioni create con successo ")

Calcolati 5,982 collegamenti
Salvati 2000 / 5982 
Salvati 4000 / 5982 
Salvati 5982 / 5982 

Relazioni create con successo 
